# Devoir maison Python
Importation des données :

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import numpy as np

df = pd.read_csv(
    'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

df.head(5)

/tmp/ipykernel_4681/868469951.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix
0,01,Ain,1,L'Abergement-Clémenciat,Nathalie,ARTHAUD,3
1,01,Ain,2,L'Abergement-de-Varey,Nathalie,ARTHAUD,2
2,01,Ain,4,Ambérieu-en-Bugey,Nathalie,ARTHAUD,38
3,01,Ain,5,Ambérieux-en-Dombes,Nathalie,ARTHAUD,8
4,01,Ain,6,Ambléon,Nathalie,ARTHAUD,0


In [90]:
!pip install great_tables
# pour great_tables

# exploration générale:
 **Question 1:**
  Création d'un vrai code commune : il faut associé le code département + le code commune sur 3 chiffres
 

In [91]:
# Construction du vrai code commune (département + commune sur 3 chiffres)
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2) +
    df['code_commune'].astype(str).str[-3:].str.zfill(3)
)

# Vérification
montrouge = df[df['libelle_commune'].str.contains('Montrouge', case=False, na=False)]
print("Code commune de Montrouge :", montrouge['code_commune'].unique())

rambouillet = df[df['libelle_commune'].str.contains('Rambouillet', case=False, na=False)]
print("Code commune de Rambouillet :", rambouillet['code_commune'].unique())

Code commune de Montrouge : ['92049']
Code commune de Rambouillet : ['78517']


Création de la variable candidat avec nom prenom ensemble avec un espace

In [92]:
# Création de la colonne candidat 
df['candidat'] = df['prenom'] + ' ' + df['nom']

# vérification 
df.sample(5)


,code_departement,libelle_departement,code_commune,libelle_commune,prenom,nom,voix,candidat
201352,64,Pyrénées-Atlantiques,64281,Jasses,Éric,ZEMMOUR,4,Éric ZEMMOUR
305612,62,Pas-de-Calais,62154,Bonnières,Yannick,JADOT,2,Yannick JADOT
465613,21,Côte-d'Or,21501,Pouilly-en-Auxois,NaN,blancs,15,NaN
19121,52,Haute-Marne,52473,Signéville,Nathalie,ARTHAUD,1,Nathalie ARTHAUD
47254,32,Gers,32174,Ladevèze-Rivière,Fabien,ROUSSEL,8,Fabien ROUSSEL


**Question 2**
On vérifie comment sont notés les votes blancs , nuls etc... puis on compte les candidats en éliminant ces votes

In [93]:
# recherche de l'écriture des votes ne correspondant pas aux candidats
df['nom'].unique()

array(['ARTHAUD', 'ROUSSEL', 'MACRON', 'LASSALLE', 'LE PEN', 'ZEMMOUR',
       'MÉLENCHON', 'HIDALGO', 'JADOT', 'PÉCRESSE', 'POUTOU',
       'DUPONT-AIGNAN', 'abstentions', 'blancs', 'nuls'], dtype=object)

In [94]:
# comptage des candidates en éliminant les abstentions etc...
candidats = df[~df['nom'].isin(['abstentions', 'blancs', 'nuls'])]['candidat'].nunique()

print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")

En 2022, il y avait 12 candidats à l'élection présidentielle.


**Question 3 **
On calcule les scores de chaque candidat.

In [95]:
# On crée une table avec uniquement les vrais candidats 
df_candidats = df[~df['nom'].isin(['abstentions', 'blancs', 'nuls'])].copy()
# On groupe les communes  par candidat et on somme sur le critère candidat 
votes= df_candidats.groupby('candidat', as_index=False)['voix'].sum()
#votes

# On somme sur toutes les voix
total_exprimes = df_candidats['voix'].sum()
#total_exprimes
#On calcule le score et on affiche l'ensemble
votes['score'] = votes['voix'] / total_exprimes
#votes

Table 1. Réponse à la question 3

In [96]:
from great_tables import GT, style, loc
# Copie pour l'affichage
table_votes = votes.sort_values('voix', ascending=False).copy()
table_votes['voix'] = table_votes['voix'].apply(lambda x: f"{x:,}".replace(',', ' '))
table_votes['score'] = table_votes['score'].apply(lambda x: f"{x:.2%}")


display(
    GT(table_votes)
    .tab_header(
        title="Elections ",
        subtitle="Résultats du premier tour du 10 avril 2022"
    )
    .cols_label(
        candidat="Candidat",
        voix="Nombre votes (total)",
        score="Score (% votes exprimés)"
    )
)

GT(_tbl_data=                 candidat       voix   score
1         Emmanuel MACRON  9 783 058  27.85%
5           Marine LE PEN  8 133 828  23.15%
4      Jean-Luc MÉLENCHON  7 712 520  21.95%
11           Éric ZEMMOUR  2 485 226   7.07%
9        Valérie PÉCRESSE  1 679 001   4.78%
10          Yannick JADOT  1 627 853   4.63%
3           Jean LASSALLE  1 101 387   3.13%
2          Fabien ROUSSEL    802 422   2.28%
7   Nicolas DUPONT-AIGNAN    725 176   2.06%
0            Anne HIDALGO    616 478   1.75%
8         Philippe POUTOU    268 904   0.77%
6        Nathalie ARTHAUD    197 094   0.56%, _body=<great_tables._gt_data.Body object at 0x7faa34a10f70>, _boxhead=Boxhead([ColInfo(var='candidat', type=<ColInfoTypeEnum.default: 1>, column_label='Candidat', column_align='left', column_width=None), ColInfo(var='voix', type=<ColInfoTypeEnum.default: 1>, column_label='Nombre votes (total)', column_align='right', column_width=None), ColInfo(var='score', type=<ColInfoTypeEnum.default: 1>, column_label='Score (% votes exprimés)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7faa34b40af0>, _spanners=Spanners([]), _heading=Heading(title='Elections ', subtitle='Résultats du premier tour du 10 avril 2022', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7faa34c04590>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7faa34c04050>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7faa34b40c00>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='t

# Comparaison des scores départements aux moyennes nationales:
 **Question 4:**
  Création d'un dataframe